# NB11 — Survival models, the H1 verdict, and per-gene inputs for tiering
**Models** (Cox, overall survival in days, continuous scores per 1 SD, no cut-points), fitted for S, M and R in each
cohort:
* m0: score alone (descriptive)
* m1: score + age + sex
* m2: m1 + two purity proxies
* m3: m2 + normal neuron-identity score (fully adjusted)

Proportional hazards: `cox.zph` p is reported and p < 0.05 is flagged. Events per parameter are reported.
Cohorts are combined by fixed-effect and DerSimonian-Laird random-effects meta-analysis (descriptive with 2–3 cohorts).

**H1 verdict** uses only the pre-specified criteria (config `survival.h1`):
* M: m3 HR > 1 in every cohort, fixed-effect meta p < 0.05, I2 < 50%;
* S: log HR changes by >= 25% between m1 and m3, or its significance at 0.05 flips;
* R: no independent adverse association (m3 meta p >= 0.05 or HR <= 1).
All three met -> `supported`; M criteria not met and the M meta CI includes 1 -> `not_supported`; otherwise `inconclusive`.

**Per-gene analysis for tiering:** the m3 model for each M gene, BH over genes, direction concordance, I2, and
leave-one-cohort-out (needs 3 cohorts).

In [ ]:
import os, sys, json
ROOT = os.environ.get("GBM_ROOT", "/content/drive/MyDrive/GBM_MultiCohort_Research")
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not in Colab; GBM_ROOT =", ROOT)
sys.path.insert(0, os.path.join(ROOT, "00_Project_Config"))
from gbm_common import *
from gbm_extra import load_expr
from scipy import stats
from statsmodels.stats.multitest import multipletests
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
stop_if_smoke("NB11 needs the full cohorts and frozen gene sets.")
verify_frozen()
VAL = P("validation"); CL = P("clinical"); MAT = P("meta", "matrices"); SV = CFG["survival"]
cohorts = [c for c in ["TCGA", "CGGA693", "CGGA325"] if (VAL / f"{c}_scores.csv").exists()]
require(len(cohorts) >= 2, f"Need at least 2 scored cohorts; found {cohorts}. Run NB09.")
r_cox = write_r("cox_run.R", R_COX)
M_genes = pd.read_csv(P("final", "M_genes.csv")).gene.astype(str).tolist()
COVS = ["purity_proxy_1", "purity_proxy_2", "neuron_identity", "immune_stromal"]


def prep(name):
    d = pd.read_csv(VAL / f"{name}_scores.csv")
    d["time"] = pd.to_numeric(d.os_time_days, errors="coerce"); d["event"] = pd.to_numeric(d.os_event, errors="coerce")
    d["sex_male"] = d.sex.astype(str).str.strip().str.lower().map(lambda s: 1.0 if s.startswith("m") else (0.0 if s.startswith("f") else np.nan))
    d["age"] = pd.to_numeric(d.age, errors="coerce")
    d = d[(d.time > 0) & d.event.isin([0, 1])].copy()
    for c in ["S", "M", "R"] + COVS:
        d[c] = (d[c] - d[c].mean()) / d[c].std()
    return d


data = {c: prep(c) for c in cohorts}
for c, d in data.items():
    print(f"{c}: n={len(d)}, events={int(d.event.sum())}, median OS days={d.time.median():.0f}, "
          f"missing age={int(d.age.isna().sum())}, missing sex={int(d.sex_male.isna().sum())}")

In [ ]:
MODELS = {"m0": "", "m1": "age,sex_male", "m2": "age,sex_male,purity_proxy_1,purity_proxy_2",
          "m3": "age,sex_male,purity_proxy_1,purity_proxy_2,neuron_identity"}
cox_rows = []
pf = CL / "pred_SMR.txt"; pf.write_text("S\nM\nR\n")
for c, d in data.items():
    dp = CL / f"{c}_cox_input.csv"; d.to_csv(dp, index=False)
    for mname, adj in MODELS.items():
        out = CL / f"{c}_cox_{mname}.csv"
        run_r(r_cox, [dp, pf, adj, out])
        t = pd.read_csv(out)
        require(len(t) == 3, f"{c} {mname}: Cox fitted for {len(t)} of 3 scores (too few complete cases or events?).")
        t["cohort"] = c; t["model"] = mname
        cox_rows.append(t)
cox = pd.concat(cox_rows, ignore_index=True)
cox["HR"] = np.exp(cox.logHR); cox["HR_lo"] = np.exp(cox.logHR - 1.96 * cox.SE); cox["HR_hi"] = np.exp(cox.logHR + 1.96 * cox.SE)
cox["events_per_param"] = cox.events / cox.n_params
cox["ph_violation"] = cox.ph_p < 0.05
cox.to_csv(CL / "cox_by_cohort.csv", index=False)
print(cox[["cohort", "model", "predictor", "n", "events", "HR", "HR_lo", "HR_hi", "p", "ph_p", "events_per_param"]].round(3).to_string(index=False))
low = cox[cox.events_per_param < SV["min_events_per_param"]]
if len(low):
    print(f"\nWARNING: {len(low)} fits have < {SV['min_events_per_param']} events per parameter (reported, not hidden).")

In [ ]:
meta_rows = []
for model in MODELS:
    for pred in ["S", "M", "R"]:
        sub = cox[(cox.model == model) & (cox.predictor == pred)].set_index("cohort").reindex(cohorts).dropna(subset=["logHR"])
        if len(sub) < 2:
            continue
        r = dl_meta(sub.logHR.values[None, :], (sub.SE.values ** 2)[None, :])
        meta_rows.append({"model": model, "predictor": pred, "n_cohorts": len(sub), "cohorts": ",".join(sub.index),
                          "logHR_fixed": r["mu_fe"][0], "SE_fixed": r["se_fe"][0],
                          "p_fixed": 2 * stats.norm.sf(abs(r["mu_fe"][0] / r["se_fe"][0])),
                          "logHR_random": r["mu"][0], "SE_random": r["se"][0], "p_random": r["p"][0], "I2": r["I2"][0], "Q": r["Q"][0],
                          "all_cohorts_HR_gt_1": bool((sub.logHR > 0).all()), "all_cohorts_HR_lt_1": bool((sub.logHR < 0).all())})
meta = pd.DataFrame(meta_rows)
meta["HR_fixed"] = np.exp(meta.logHR_fixed)
meta["HR_fixed_lo"] = np.exp(meta.logHR_fixed - 1.96 * meta.SE_fixed); meta["HR_fixed_hi"] = np.exp(meta.logHR_fixed + 1.96 * meta.SE_fixed)
meta["HR_random"] = np.exp(meta.logHR_random)
meta.to_csv(CL / "cox_meta_analysis.csv", index=False)
print(meta[["model", "predictor", "n_cohorts", "HR_fixed", "HR_fixed_lo", "HR_fixed_hi", "p_fixed", "HR_random", "p_random", "I2"]].round(3).to_string(index=False))

In [ ]:
# ---- H1 verdict --------------------------------------------------------------------------------------------
H = SV["h1"]


def g(m, p):
    x = meta[(meta.model == m) & (meta.predictor == p)]
    return x.iloc[0] if len(x) else None


m3M, m1S, m3S, m3R = g("m3", "M"), g("m1", "S"), g("m3", "S"), g("m3", "R")
reasons, verdict, flags = [], "inconclusive", {}
if any(x is None for x in [m3M, m1S, m3S, m3R]):
    reasons.append("meta-analysis not evaluable (fewer than 2 cohorts fitted)")
else:
    M_ok = bool(m3M.all_cohorts_HR_gt_1 and m3M.p_fixed < H["meta_p"] and m3M.I2 < H["i2_max"])
    shift = abs(m3S.logHR_fixed - m1S.logHR_fixed) / max(abs(m1S.logHR_fixed), 1e-9)
    flip = bool((m1S.p_fixed < 0.05) != (m3S.p_fixed < 0.05))
    S_ok = bool(shift >= H["hr_shift_frac"] or flip)
    R_ok = bool(m3R.p_fixed >= 0.05 or m3R.logHR_fixed <= 0)
    flags = {"M_criteria_met": M_ok, "S_criteria_met": S_ok, "R_criteria_met": R_ok, "S_shift_fraction": float(shift), "S_significance_flip": flip}
    reasons += [f"M criteria met: {M_ok} (m3 HR={m3M.HR_fixed:.2f} [{m3M.HR_fixed_lo:.2f}-{m3M.HR_fixed_hi:.2f}], p={m3M.p_fixed:.3g}, "
                f"I2={m3M.I2:.0f}%, all cohorts HR>1: {m3M.all_cohorts_HR_gt_1})",
                f"S change m1->m3 = {shift * 100:.0f}% (needs >= {H['hr_shift_frac'] * 100:.0f}%) or significance flip = {flip}: met = {S_ok} "
                f"(m1 HR={m1S.HR_fixed:.2f}, m3 HR={m3S.HR_fixed:.2f})",
                f"R lacks independent adverse association: {R_ok} (m3 HR={m3R.HR_fixed:.2f}, p={m3R.p_fixed:.3g})"]
    if M_ok and S_ok and R_ok:
        verdict = "supported"
    elif (not M_ok) and m3M.HR_fixed_lo <= 1:
        verdict = "not_supported"
ph_bad = cox[(cox.model == "m3") & cox.ph_violation]
if len(ph_bad):
    reasons.append(f"proportional-hazards flag (cox.zph p<0.05) in m3 for: {', '.join(ph_bad.cohort + ':' + ph_bad.predictor)}")
res = {"verdict": verdict, "reasons": reasons, "flags": flags, "cohorts": cohorts, "criteria": H,
       "n_per_cohort": {c: int(len(d)) for c, d in data.items()}, "events_per_cohort": {c: int(d.event.sum()) for c, d in data.items()},
       "note": "Association only; marker-score purity proxies are imperfect; see limitations."}
(CL / "H1_verdict.json").write_text(json.dumps(res, indent=2, default=float))
manifest_update("H1", "verdict", res)
print("\nH1 VERDICT:", verdict.upper())
for r_ in reasons:
    print(" -", r_)

In [ ]:
def hr_axis(ax):
    """Readable log-scale HR axis: plain numbers, no overlapping minor labels."""
    from matplotlib.ticker import FixedLocator, FuncFormatter, NullLocator
    lo, hi = ax.get_xlim()
    ticks = [t for t in [0.25, 0.5, 0.75, 1, 1.5, 2, 3, 4, 6, 8] if lo <= t <= hi] or [1]
    ax.xaxis.set_major_locator(FixedLocator(ticks)); ax.xaxis.set_minor_locator(NullLocator())
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))


# ---- forest plot (per cohort + fixed-effect meta, m1 vs m3) ---------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharex=True)
for ax, pred in zip(axes, ["S", "M", "R"]):
    rows = []
    for model, col in [("m1", "#9aa5b1"), ("m3", "#1f3a93")]:
        for c in cohorts:
            x = cox[(cox.model == model) & (cox.predictor == pred) & (cox.cohort == c)]
            if len(x):
                rows.append((f"{c} {model}", x.HR.iloc[0], x.HR_lo.iloc[0], x.HR_hi.iloc[0], col))
        mm = g(model, pred)
        if mm is not None:
            rows.append((f"Meta {model}", mm.HR_fixed, mm.HR_fixed_lo, mm.HR_fixed_hi, "#c0392b" if model == "m3" else "#e6a0a0"))
    for i, (lab, hr, lo, hi, col) in enumerate(rows[::-1]):
        ax.errorbar(hr, i, xerr=[[hr - lo], [hi - hr]], fmt="o", color=col, ms=4, capsize=2)
    ax.set_yticks(range(len(rows))); ax.set_yticklabels([r[0] for r in rows[::-1]], fontsize=7)
    ax.axvline(1, c="k", lw=.6, ls=":"); ax.set_xscale("log"); hr_axis(ax); ax.set_title(f"{pred} score (HR per SD)")
fig.tight_layout(); savefig(fig, "Forest_S_M_R_m1_vs_m3", folder="clinical"); plt.close(fig)

In [ ]:
# ---- per-gene Cox (m3) for M genes ------------------------------------------------------------------------
gene_cox = []
for c in cohorts:
    fn = MAT / ("TCGA_logcpm.csv.gz" if c == "TCGA" else f"{c}_log2fpkm.csv.gz")
    e = load_expr(fn)
    ids = [s for s in data[c].sample_id if s in e.columns]
    genes = [x for x in M_genes if x in e.index]
    z = zscore_df(e.loc[genes, ids], axis=1).T
    z = z.loc[:, z.std() > 0]
    d = data[c].set_index("sample_id").loc[ids, ["time", "event", "age", "sex_male", "purity_proxy_1", "purity_proxy_2", "neuron_identity"]]
    full = pd.concat([d, z], axis=1)
    dp = CL / f"{c}_gene_cox_input.csv.gz"; full.to_csv(dp, index=False)
    pfg = CL / f"{c}_gene_preds.txt"; pfg.write_text("\n".join(z.columns))
    out = CL / f"{c}_gene_cox_m3.csv"
    print(c, run_r(r_cox, [dp, pfg, MODELS["m3"], out]).strip())
    t = pd.read_csv(out); t["cohort"] = c; gene_cox.append(t)
gc_ = pd.concat(gene_cox)
gc_.to_csv(CL / "gene_cox_by_cohort.csv", index=False)

wide_b = gc_.pivot(index="predictor", columns="cohort", values="logHR")
wide_s = gc_.pivot(index="predictor", columns="cohort", values="SE")
ok = wide_b[cohorts].dropna().index
Yb, Vb = wide_b.loc[ok, cohorts].values, wide_s.loc[ok, cohorts].values ** 2
r = dl_meta(Yb, Vb)
tab = pd.DataFrame({"gene": ok, "meta_logHR": r["mu"], "meta_SE": r["se"], "meta_p": r["p"], "I2": r["I2"],
                    "fixed_logHR": r["mu_fe"], "fixed_p": 2 * stats.norm.sf(np.abs(r["mu_fe"] / r["se_fe"]))})
tab["meta_FDR"] = multipletests(tab.meta_p, method="fdr_bh")[1]
tab["direction_concordant"] = (np.sign(Yb) == np.sign(Yb[:, [0]])).all(axis=1)
for j, c in enumerate(cohorts):
    tab[f"logHR_{c}"] = Yb[:, j]
if len(cohorts) >= 3:
    loo_ok = np.ones(len(tab), bool)
    for j in range(len(cohorts)):
        keep = [i for i in range(len(cohorts)) if i != j]
        rr = dl_meta(Yb[:, keep], Vb[:, keep])
        fdr = multipletests(rr["p"], method="fdr_bh")[1]
        tab[f"LOO_FDR_without_{cohorts[j]}"] = fdr
        loo_ok &= (fdr < CFG["tiers"]["fdr"]) & (np.sign(rr["mu"]) == np.sign(r["mu"]))
    tab["loo_stable"] = loo_ok
else:
    tab["loo_stable"] = np.nan
    log("Leave-one-cohort-out needs >= 3 cohorts; component recorded as not evaluable.", "WARN")
tab = tab.sort_values("meta_p")
tab.to_csv(CL / "gene_cox_meta_M_genes.csv", index=False)
print(f"M genes with Cox in all cohorts: {len(tab)} | meta FDR<0.05: {int((tab.meta_FDR < 0.05).sum())} | "
      f"concordant: {int(tab.direction_concordant.sum())}")
print(tab.head(10)[["gene", "meta_logHR", "meta_FDR", "I2", "direction_concordant", "loo_stable"]].round(4).to_string(index=False))
print("\nNB11 complete. Next: NB08 (literature), NB12, NB13, NB14, NB15.")